# 1. Install Requriments

In [1]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 801.7 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 458.3 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 677.7 kB/s eta 0:00:000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 584.3 kB/s eta 0:00:00a 0:00:01
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 667.7 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 2.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 6.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 23.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.

# 2. Import Libraries

In [2]:
import pandas as pd
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, TrainingArguments, logging
import huggingface_hub
from trl import SFTTrainer
import re

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[torchao|WARNING]Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Login to Hugging Face
huggingface_hub.login(HF_TOKEN)

# Load dataset
df_train = pd.read_csv('train_sample_prompt.csv')
df_val = pd.read_csv('valid_sample_prompt.csv')
dataset = Dataset.from_pandas(df_train)
val_dataset = Dataset.from_pandas(df_val)

# Load model dan tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Yellow-AI-NLP/komodo-7b-base",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
    token=True,
    trust_remote_code=True
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.10.11: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla V100-PCIE-16GB. Num GPUs = 1. Max memory: 15.766 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.39M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/58.1k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Yellow-AI-NLP/komodo-7b-base does not have a padding token! Will use pad_token = <unk>.


# 3. Fine Tuning Using QLoRA

In [4]:
def train(model, tokenizer, rank, alpha, dropout):
    # Apply LoRA (hanya q_proj dan v_proj)
    model = FastLanguageModel.get_peft_model(
        model,
        target_modules=["q_proj", "v_proj"],
        r=rank,
        lora_alpha=alpha,
        lora_dropout=dropout,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        max_seq_length=None,
    )
    
    # Default TrainingArguments dengan penyesuaian minimal
    training_arguments = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=4, # standar 8
        per_device_eval_batch_size=4, # standar 8
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        logging_steps=100,
        learning_rate=5e-5,  # default Transformers
        weight_decay=0.0, # default 
        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="tensorboard",
    )
    
    # Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        eval_dataset=val_dataset,
        dataset_text_field="text",
        tokenizer=tokenizer,
        args=training_arguments,
        packing=False,
    )
    
    # Training
    trainer.train()
    # Save best model
    
    trainer.model.save_pretrained(f"Komodo-7b-summarize-best-{rank}-{alpha}-{dropout}")
    tokenizer.save_pretrained(f"Komodo-7b-summarize-best-{rank}-{alpha}-{dropout}")

    log_history = trainer.state.log_history  # ambil semua log (loss, eval_loss, dll)
    log_df = pd.DataFrame(log_history)
    log_df.to_csv(f"hasil-training-{rank}-{alpha}-{dropout}.csv", index=False)

In [5]:
rank = 32
alpha = 32
dropout = 0.05
train(model, tokenizer, rank, alpha, dropout)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.11 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=54):   0%|          | 0/9000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=54):   0%|          | 0/500 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,000 | Num Epochs = 3 | Total steps = 6,750
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 16,777,216 of 6,779,834,368 (0.25% trained)


Step,Training Loss,Validation Loss
200,1.006800,1.042714
400,0.980700,1.033848
600,0.950200,1.027674
800,0.956300,1.024256
1000,0.954000,1.022158
1200,0.945900,1.016975
1400,0.944200,1.014468
1600,0.948300,1.014489
1800,0.956000,1.014058
2000,0.952600,1.013019


Unsloth: Will smartly offload gradients to save VRAM!
